In [2]:
import zipfile
import pandas as pd
import os
from datetime import datetime
import json

# Path to your file
file_path = '../data/raw/KL_dataset.xlsx'

# ============================================
# 1. BASIC FILE INFORMATION
# ============================================
print("="*60)
print("FILE INFORMATION")
print("="*60)

file_stats = os.stat(file_path)
print(f"File name: {os.path.basename(file_path)}")
print(f"File size: {file_stats.st_size} bytes ({file_stats.st_size/1024/1024:.2f} MB)")
print(f"Created: {datetime.fromtimestamp(file_stats.st_ctime)}")
print(f"Modified: {datetime.fromtimestamp(file_stats.st_mtime)}")
print(f"Actual type: ZIP archive (PK signature detected)")

# ============================================
# 2. EXPLORE ZIP CONTENTS
# ============================================
print("\n" + "="*60)
print("ZIP ARCHIVE CONTENTS")
print("="*60)

with zipfile.ZipFile(file_path, 'r') as zip_ref:
    # Get all files info
    for file_info in zip_ref.infolist():
        print(f"\n📁 File: {file_info.filename}")
        print(f"   Size: {file_info.file_size} bytes ({file_info.file_size/1024:.2f} KB)")
        print(f"   Compressed: {file_info.compress_size} bytes")
        print(f"   Compression ratio: {(1 - file_info.compress_size/file_info.file_size)*100:.1f}%")
        print(f"   Modified: {datetime(*file_info.date_time)}")
    
    # List all files
    print("\n" + "-"*40)
    print("All files in archive:")
    for name in zip_ref.namelist():
        print(f"  • {name}")

# ============================================
# 3. EXTRACT AND ANALYZE THE ACTUAL DATA
# ============================================
print("\n" + "="*60)
print("DATA CONTENT ANALYSIS")
print("="*60)

# Extract and read the data
with zipfile.ZipFile(file_path, 'r') as zip_ref:
    # Find CSV files
    csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
    
    if csv_files:
        print(f"\nFound {len(csv_files)} CSV file(s):")
        
        for csv_file in csv_files:
            print(f"\n📊 Analyzing: {csv_file}")
            print("-"*40)
            
            # Read the CSV
            with zip_ref.open(csv_file) as f:
                # Try different encodings
                for encoding in ['utf-8', 'latin1', 'cp1252', 'iso-8859-1']:
                    try:
                        df = pd.read_csv(f, encoding=encoding)
                        print(f"✓ Successfully read with encoding: {encoding}")
                        break
                    except:
                        f.seek(0)  # Reset file pointer
                        continue
                
                # DATAFRAME METADATA
                print(f"\n📈 Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
                
                print(f"\n📋 Column Information:")
                for col in df.columns:
                    print(f"  • {col}: {df[col].dtype} ({(df[col].count()/len(df))*100:.1f}% complete)")
                
                print(f"\n📊 Data Types Summary:")
                print(df.dtypes.value_counts())
                
                print(f"\n🔢 Memory Usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")
                
                print(f"\n👀 First 5 rows:")
                print(df.head())
                
                print(f"\n📑 Last 5 rows:")
                print(df.tail())
                
                print(f"\n📊 Basic Statistics (numeric columns):")
                print(df.describe())
                
                print(f"\n🔍 Missing Values:")
                missing = df.isnull().sum()
                missing = missing[missing > 0]
                if len(missing) > 0:
                    print(missing)
                else:
                    print("No missing values found!")
                
                print(f"\n🔄 Unique Values per Column:")
                for col in df.columns:
                    unique_count = df[col].nunique()
                    print(f"  • {col}: {unique_count} unique values")
                
                # Save metadata to file
                metadata = {
                    'file_name': os.path.basename(file_path),
                    'file_size_mb': file_stats.st_size/1024/1024,
                    'dataset_name': csv_file,
                    'shape': {'rows': df.shape[0], 'columns': df.shape[1]},
                    'columns': list(df.columns),
                    'dtypes': df.dtypes.astype(str).to_dict(),
                    'missing_values': df.isnull().sum().to_dict(),
                    'unique_counts': df.nunique().to_dict(),
                    'memory_usage_kb': df.memory_usage(deep=True).sum() / 1024
                }
                
                # Save metadata as JSON
                with open('../data/raw/dataset_metadata.json', 'w') as json_file:
                    json.dump(metadata, json_file, indent=2)
                print(f"\n💾 Metadata saved to: dataset_metadata.json")
                
                # Assign to variable for later use
                cbc_data = df
    else:
        print("No CSV files found in the archive")

# ============================================
# 4. CHECK FOR OTHER DATA FORMATS
# ============================================
print("\n" + "="*60)
print("ADDITIONAL FILES IN ARCHIVE")
print("="*60)

with zipfile.ZipFile(file_path, 'r') as zip_ref:
    # Check for Excel files
    excel_files = [f for f in zip_ref.namelist() if f.endswith(('.xlsx', '.xls'))]
    if excel_files:
        print(f"\n📊 Excel files found: {excel_files}")
    
    # Check for JSON files
    json_files = [f for f in zip_ref.namelist() if f.endswith('.json')]
    if json_files:
        print(f"\n📋 JSON files found: {json_files}")
    
    # Check for text files
    txt_files = [f for f in zip_ref.namelist() if f.endswith('.txt')]
    if txt_files:
        print(f"\n📝 Text files found: {txt_files}")

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)

FILE INFORMATION
File name: KL_dataset.xlsx
File size: 45864858 bytes (43.74 MB)
Created: 2026-06-17 01:48:04.484389
Modified: 2026-06-17 01:48:04.505974
Actual type: ZIP archive (PK signature detected)

ZIP ARCHIVE CONTENTS

📁 File: [Content_Types].xml
   Size: 1284 bytes (1.25 KB)
   Compressed: 366 bytes
   Compression ratio: 71.5%
   Modified: 1980-01-01 00:00:00

📁 File: _rels/.rels
   Size: 588 bytes (0.57 KB)
   Compressed: 244 bytes
   Compression ratio: 58.5%
   Modified: 1980-01-01 00:00:00

📁 File: xl/_rels/workbook.xml.rels
   Size: 698 bytes (0.68 KB)
   Compressed: 243 bytes
   Compression ratio: 65.2%
   Modified: 1980-01-01 00:00:00

📁 File: xl/workbook.xml
   Size: 716 bytes (0.70 KB)
   Compressed: 401 bytes
   Compression ratio: 44.0%
   Modified: 1980-01-01 00:00:00

📁 File: xl/sharedStrings.xml
   Size: 433 bytes (0.42 KB)
   Compressed: 228 bytes
   Compression ratio: 47.3%
   Modified: 1980-01-01 00:00:00

📁 File: xl/worksheets/_rels/sheet1.xml.rels
   Size: 322 

In [3]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

# Read the Excel file
df = pd.read_excel('../data/raw/KL_dataset.xlsx')

# View your data
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nLast 5 rows:")
print(df.tail())
print(f"\nData types:")
print(df.dtypes)
print(f"\nBasic info:")
df.info()

Dataset shape: (523844, 13)

Columns: ['SampleNum', 'Timestamp', 'PatientNum', 'WardNum', 'ERY', 'HK', 'LEUKO', 'HB', 'PLT', 'MCV', 'MCHC', 'MCH', 'RDW']

First 5 rows:
   SampleNum           Timestamp  PatientNum  WardNum  ERY    HK  LEUKO    HB  \
0  601978348 2019-12-23 07:43:16    13519502        1  4.5  40.7    8.5  13.7   
1  607541209 2019-12-10 10:22:16    13495186        2  4.2  36.8   11.8  12.9   
2  606368887 2019-12-06 04:47:16    13827037        3  4.3  37.4    8.5  12.7   
3  603522458 2019-11-29 10:38:16    13827037        4  4.3  36.4    7.7  13.3   
4  601574703 2019-11-26 04:47:16    13511555        5  4.1  37.1   10.8  12.3   

   PLT   MCV  MCHC   MCH   RDW  
0  154  90.4  33.7  30.4  13.2  
1  255  87.4  35.1  30.6  12.7  
2  310  86.4  33.9  29.3  13.0  
3  221  84.4  36.6  30.9  13.1  
4  198  91.3  33.2  30.3  13.5  

Last 5 rows:
        SampleNum           Timestamp  PatientNum  WardNum  ERY    HK  LEUKO  \
523839  603010564 2024-03-22 06:46:16    13268285   

In [5]:
# Informations générales : types de colonnes, valeurs non-nulles
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 523844 entries, 0 to 523843
Data columns (total 13 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   SampleNum   523844 non-null  int64         
 1   Timestamp   523844 non-null  datetime64[ns]
 2   PatientNum  523844 non-null  int64         
 3   WardNum     523844 non-null  int64         
 4   ERY         523844 non-null  float64       
 5   HK          523844 non-null  float64       
 6   LEUKO       523844 non-null  float64       
 7   HB          523844 non-null  float64       
 8   PLT         523844 non-null  int64         
 9   MCV         523844 non-null  float64       
 10  MCHC        523844 non-null  float64       
 11  MCH         523844 non-null  float64       
 12  RDW         523844 non-null  float64       
dtypes: datetime64[ns](1), float64(8), int64(4)
memory usage: 52.0 MB


In [6]:
# Statistiques descriptives
df.describe(include='all')

,SampleNum,Timestamp,PatientNum,WardNum,ERY,HK,LEUKO,HB,PLT,MCV,MCHC,MCH,RDW
count,5.238440e+05,523844,5.238440e+05,523844.000000,523844.000000,523844.000000,523844.000000,523844.000000,523844.000000,523844.000000,523844.000000,523844.000000,523844.000000
mean,6.049975e+08,2022-07-08 08:56:52.517434368,1.349790e+07,138.723152,4.107627,37.022095,9.025179,12.273387,255.375079,90.695432,33.134883,30.051981,14.891602
min,1.343095e+07,2019-11-23 21:22:16,1.300000e+07,1.000000,0.700000,5.700000,0.100000,2.100000,3.000000,37.500000,23.100000,12.000000,10.000000
25%,6.024882e+08,2021-09-03 03:41:31,1.324884e+07,23.000000,3.600000,32.600000,6.100000,10.700000,186.000000,86.500000,32.400000,28.700000,13.500000
50%,6.049938e+08,2022-07-16 07:18:16,1.349753e+07,68.000000,4.100000,37.300000,8.000000,12.400000,238.000000,90.500000,33.200000,30.200000,14.400000
75%,6.075070e+08,2023-05-21 05:45:16,1.374799e+07,211.000000,4.600000,41.600000,10.600000,13.900000,303.000000,94.700000,33.900000,31.500000,15.700000
max,6.100000e+08,2024-03-23 04:47:16,1.399998e+07,685.000000,10.000000,84.500000,571.700000,28.100000,2689.000000,144.700000,152.100000,104.800000,40.300000
std,3.002106e+06,NaN,2.889726e+05,163.582425,0.754331,6.521979,6.268226,2.235249,112.847395,7.423353,1.270782,2.648766,2.010604


In [7]:
# Liste des colonnes
print(df.columns.tolist())

['SampleNum', 'Timestamp', 'PatientNum', 'WardNum', 'ERY', 'HK', 'LEUKO', 'HB', 'PLT', 'MCV', 'MCHC', 'MCH', 'RDW']


In [8]:
# Vérifier les valeurs manquantes
df.isnull().sum()

SampleNum     0
Timestamp     0
PatientNum    0
WardNum       0
ERY           0
HK            0
LEUKO         0
HB            0
PLT           0
MCV           0
MCHC          0
MCH           0
RDW           0
dtype: int64

In [9]:
# Vérifier les doublons
print("Doublons:", df.duplicated().sum())

Doublons: 0
